# Mating regimes

A mating regime decides **who pairs with whom** and **how many
offspring each pair produces**. The v0.9 mating regimes live in
`xftsim.mate`:

| Class | Behaviour |
| --- | --- |
| `RandomMating` | Sex-balanced random pairing |
| `LinearAssortativeMating` | Exchangeable assortative mating on a phenotypic composite |
| `GeneralAssortativeMating` | Arbitrary `K x K` target cross-mate correlation matrix via QAP (needs Hexaly) |
| `BatchedMating` | Wraps any regime and applies it to random partitions of the sample |

Naming and API differences vs the legacy interface:

- Suffix `Regime` has been dropped (`LinearAssortativeMatingRegime` →
  `LinearAssortativeMating`, etc.).
- Component selection takes a `component_names=[str, ...]` list rather
  than a `xftsim.index.ComponentIndex`.
- `mates_per_female`, `female_offspring_per_pair`, `sex_aware`, and
  `exhaustive` are no longer constructor arguments. Sex-balanced
  pairing is always on; one female pairs with one male; the only
  remaining count knob is `offspring_per_pair: int`.
- `xftsim.utils.VariableCount` (and its subclasses `ConstantCount`,
  `PoissonCount`, `NegativeBinomialCount`, `ZeroTruncatedPoissonCount`,
  `MixtureCount`) is **no longer used by the mating regimes**. The
  classes still exist in `xftsim.utils`, but if you want variable
  offspring counts you'll need to drive that yourself.

We'll go through the regimes in turn. We'll first build a small
shared simulation to use as a backbone.


In [ ]:
import xftsim as xft
import numpy as np

xft.config.print_durations_threshold = 10.
np.random.seed(123)

# Reusable two-trait setup for the rest of the tutorial.
N, M = 1000, 200
hap = xft.founders.founder_haplotypes_uniform_AFs(n=N, m=M)
eff_h = xft.effect.AdditiveEffects.from_h2(h2=0.6, m=M, seed=1)
eff_b = xft.effect.AdditiveEffects.from_h2(h2=0.4, m=M, seed=2)
arch = xft.arch.Architecture(
    formula='''
    height.G ~ genetic(eff_h)
    height.E ~ noise(0.4)
    height   ~ height.G + height.E

    BMD.G ~ genetic(eff_b)
    BMD.E ~ noise(0.6)
    BMD   ~ BMD.G + BMD.E
    ''',
    effects={'eff_h': eff_h, 'eff_b': eff_b},
)
rmap = xft.reproduce.RecombinationMap.from_haplotypes(hap, p=0.1)


## Random mating

`RandomMating` pairs individuals sex-balanced and uniformly at random:


In [ ]:
mating_rm = xft.mate.RandomMating(offspring_per_pair=2)

sim_rm = xft.sim.Simulation(
    founder_haplotypes=hap,
    architecture=arch,
    recombination_map=rmap,
    mating_regime=mating_rm,
    statistics=[xft.stats.SampleStatistics()],
    seed=42,
)
sim_rm.run(n_generations=1)
sim_rm.results[-1].statistics['SampleStatistics']['keys']


## Mate assignments

A mating regime maps the current `SampleMeta` (and optionally the
current `PhenotypeArray`) to a `MateAssignment` dataclass. You'll
rarely call `.mate()` yourself — the simulation does it each
generation — but it's useful to know the shape:


In [ ]:
rng = np.random.RandomState(0)
ma = mating_rm.mate(hap.samples, rng=rng)
print(ma)
print('first 5 mother indices:', ma.maternal_idx[:5])
print('first 5 father indices:', ma.paternal_idx[:5])


The legacy `MateAssignment.get_mating_frame()` /
`get_reproduction_frame()` helpers are gone — the assignment is just
two integer index arrays into the parent generation, plus a
`SampleMeta` describing the offspring.

## Linear assortative mating

`LinearAssortativeMating` constructs a phenotypic composite by
averaging the standardised values of the chosen components and
rank-orders both sexes on a noised version of that composite,
producing a target spousal correlation `r`.

### Univariate primary-phenotype assortment

We can assort on a single phenotype by listing its outcome name:


In [ ]:
mating_pp = xft.mate.LinearAssortativeMating(
    component_names=['height'],
    r=0.5,
    offspring_per_pair=2,
)

sim_pp = xft.sim.Simulation(
    founder_haplotypes=hap,
    architecture=arch,
    recombination_map=rmap,
    mating_regime=mating_pp,
    statistics=[xft.stats.SampleStatistics(),
                xft.stats.MatingStatistics()],
    filters={'trio': xft.filters.TrioFilter()},
    seed=42,
)
# Run for 2 generations: spouse correlations are computed via TrioView,
# which is empty at generation 0. The gen-1 row reflects the founder pairing.
sim_pp.run(n_generations=2)
sim_pp.results[-1].statistics['MatingStatistics']['spouse_correlations']

### Univariate social / genetic homogamy

Assorting on the noise or genetic component of a phenotype instead of
the outcome reproduces social or genetic homogamy:


In [ ]:
mating_social = xft.mate.LinearAssortativeMating(
    component_names=['height.E'], r=0.5,
)
mating_genetic = xft.mate.LinearAssortativeMating(
    component_names=['height.G'], r=0.5,
)


### Bivariate exchangeable assortment

To assort jointly on multiple phenotypes (still with an exchangeable
correlation target), list all of them:


In [ ]:
mating_biv = xft.mate.LinearAssortativeMating(
    component_names=['height', 'BMD'],
    r=0.1,
)
sim_biv = xft.sim.Simulation(
    founder_haplotypes=hap,
    architecture=arch,
    recombination_map=rmap,
    mating_regime=mating_biv,
    statistics=[xft.stats.SampleStatistics(),
                xft.stats.MatingStatistics()],
    filters={'trio': xft.filters.TrioFilter()},
    seed=42,
)
sim_biv.run(n_generations=2)
sim_biv.results[-1].statistics['MatingStatistics']['spouse_correlations']

## Generalized assortative mating

`GeneralAssortativeMating` lets you target an arbitrary `K x K`
cross-mate cross-trait correlation matrix `Omega`. Under the hood the
problem is a [Quadratic Assignment Problem](https://en.wikipedia.org/wiki/Quadratic_assignment_problem)
solved approximately with the [Hexaly Optimizer](https://www.hexaly.com).

:::{warning}
Hexaly is a separately-installed dependency and requires a license
(free academic licenses are available).
:::

For example, the UK-Biobank-inspired cross-mate matrix from
[Border et al. (2022)](https://doi.org/10.1126/science.abo2059) over
BMI, height, education, and smoking:


In [ ]:
xmate_corr = np.array([
    [+0.26, -0.05, -0.11, +0.08],
    [-0.05, +0.24, +0.10, -0.02],
    [-0.11, +0.10, +0.33, -0.06],
    [+0.08, -0.02, -0.06, +0.19],
])

# (illustrative — requires hexaly to actually run)
mating_general = xft.mate.GeneralAssortativeMating(
    component_names=['bmi', 'height', 'edu', 'smoke'],
    cross_corr=xmate_corr,
    offspring_per_pair=2,
    solver_params=dict(nb_threads=8, time_limit=120,
                       time_between_displays=5),
)


The legacy `control=dict(...)` kwarg has been renamed `solver_params`.

## Batched mating

`BatchedMating` wraps any regime and applies it independently to
random partitions of the sample of at most `max_batch_size`
individuals each. This is essential for `GeneralAssortativeMating` on
large samples, because the QAP solver scales quadratically with batch
size.


In [ ]:
inner = xft.mate.GeneralAssortativeMating(
    component_names=['bmi', 'height', 'edu', 'smoke'],
    cross_corr=xmate_corr,
    solver_params=dict(nb_threads=8, time_limit=30),
)
batched = xft.mate.BatchedMating(inner, max_batch_size=1000)


In larger simulations this is the only practical way to apply
generalized assortative mating.
